In [9]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import itertools 

In [10]:
kw = 8.8581 #Angstroms

In [18]:
import itertools
import numpy as np

def find_all_hkl_cubic(target, allow_negatives=False):
    """
    Find the set of Miller indicies that satisfy h**2 + k**2 + l**2 = target, 
    which is the set of miller indicies that satisfy the bragg condition for a given target.
    """
    target = int(target)
    
    solutions = []
    
    max_val = int(np.sqrt(target))
    
    val_range = range(-max_val, max_val + 1) if allow_negatives else range(0, max_val + 1)
    
    for h, k, l in itertools.product(val_range, repeat=3):
        if h**2 + k**2 + l**2 == target:
            solutions.append((h, k, l))
            
    return sorted(list(set(solutions)), reverse=True)

def get_cubic_a(miller_idx, theta):
    h, k, l = miller_idx
    return (np.pi / (kw * np.sin(theta))) * np.sqrt(h**2 + k**2 + l**2)


def calc_d_hkl(theta, kw=kw):
    #calculate the lattice plane spacing given a bragg angle, and wave vector kw
    d_hkl = np.pi / (kw * np.sin(theta))
    return d_hkl

In [19]:
df = pd.read_csv("data.csv")

df['alpha_A'] = df['A']
df['alpha_B'] = df['B']
df.drop(columns=['A', 'B'], inplace=True)

# theta is the half angle of the diffraction angle, which is what we need for Bragg's law.
df['theta_A'] = df['alpha_A'] / 2
df['theta_B'] = df['alpha_B'] / 2

df['dhkl_A'] = calc_d_hkl(df['theta_A'])
df['dhkl_B'] = calc_d_hkl(df['theta_B'])

df['sinqr_theta_A'] = np.sin(df['theta_A'])**2
df['sinqr_theta_B'] = np.sin(df['theta_B'])**2



df['norm_sinqr_theta_A'] = np.round(df['sinqr_theta_A'] / df['sinqr_theta_A'][0]).astype(int)  ## this is definately cubic since everything is within 1e-4 of an integer. 
df['norm_sinqr_theta_B'] = (df['sinqr_theta_B'] / df['sinqr_theta_B'][0])

df['hkl_A'] = df['norm_sinqr_theta_A'].apply(find_all_hkl_cubic) #miller indicies are easier to find when cubic

df

,i,alpha_A,alpha_B,theta_A,theta_B,dhkl_A,dhkl_B,sinqr_theta_A,sinqr_theta_B,norm_sinqr_theta_A,norm_sinqr_theta_B,hkl_A
0,1,0.351821,0.287817,0.175910,0.143908,2.026561,2.472993,0.030627,0.020567,1,1.000000,"[(1, 0, 0), (0, 1, 0), (0, 0, 1)]"
1,2,0.500185,0.309260,0.250092,0.154630,1.432997,2.302754,0.061253,0.023720,2,1.153322,"[(1, 1, 0), (1, 0, 1), (0, 1, 1)]"
2,3,0.615923,0.341434,0.307961,0.170717,1.170037,2.087585,0.091880,0.028862,3,1.403323,"[(1, 1, 1)]"
3,4,0.715162,0.424062,0.357581,0.212031,1.013281,1.685268,0.122506,0.044287,4,2.153317,"[(2, 0, 0), (0, 2, 0), (0, 0, 2)]"
4,5,0.804135,0.535513,0.402068,0.267757,0.906307,1.340513,0.153133,0.069997,5,3.403327,"[(2, 1, 0), (2, 0, 1), (1, 2, 0), (1, 0, 2), (..."
5,6,0.886044,0.540061,0.443022,0.270031,0.827341,1.329497,0.183759,0.071161,6,3.459962,"[(2, 1, 1), (1, 2, 1), (1, 1, 2)]"
6,7,0.962795,0.581821,0.481397,0.290911,0.765969,1.236497,0.214386,0.082268,7,4.000000,[]
7,8,1.035640,0.615399,0.517820,0.307700,0.716499,1.171001,0.245012,0.091728,8,4.459961,"[(2, 2, 0), (2, 0, 2), (0, 2, 2)]"
8,9,1.105460,0.626241,0.552730,0.313121,0.675522,1.151378,0.275638,0.094882,9,4.613284,"[(3, 0, 0), (2, 2, 1), (2, 1, 2), (1, 2, 2), (..."
9,10,1.172910,0.643580,0.586455,0.321790,0.640857,1.121393,0.306265,0.100024,10,4.863288,"[(3, 1, 0), (3, 0, 1), (1, 3, 0), (1, 0, 3), (..."


In [ ]:

a_values = []

for idx in range(len(df)):
    theta = df['theta_A'][idx]
    
    for hkl in df['hkl_A'][idx]:
        a = get_cubic_a(hkl, theta)
        a_values.append(a)

a_array = np.array(a_values)

average_a = np.mean(a_array)
variance_a = np.var(a_array)

print(f"Total permutations calculated: {len(a_array)}")
print(f"Average lattice parameter 'a': {average_a:.6f} Å")
print(f"Variance of 'a': {variance_a:.4e} Å²")

Total permutations calculated: 47
Average lattice parameter 'a': 2.026564 Å
Variance of 'a': 4.9421e-12 Å²


In [14]:
df['theta_A'][0]

np.float64(0.1759105)